In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import roll_time_series


1) import data -> 250ms_labeled.pkl -> df
2) from df, select only the 'excercises' that are going to be analysed 

In [3]:
df = pd.read_pickle('../data/03-features/250MS_labeled.pkl')

In [4]:
mask_targetexercises = (df['target_name'] == 'bench pullover') | (df['target_name'] == 'bicep curl') | (df['target_name'] == 'dumbbell press') | (df['target_name'] == 'pecfly') | (df['target_name'] == 'pullup') | (df['target_name'] == 'tricep over head')

In [5]:
df_targetexercise = df[mask_targetexercises].copy()

In [6]:
df_targetexercise.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6586 entries, 2025-12-12 14:48:31 to 2025-12-27 16:59:22.250000
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   HR_bpm              6586 non-null   float64
 1   ACC_z               6586 non-null   float64
 2   ACC_y               6586 non-null   float64
 3   ACC_x               6586 non-null   float64
 4   G_z                 6586 non-null   float64
 5   G_y                 6586 non-null   float64
 6   G_x                 6586 non-null   float64
 7   GYRO_z              6586 non-null   float64
 8   GYRO_y              6586 non-null   float64
 9   GYRO_x              6586 non-null   float64
 10  OR_yaw              6586 non-null   float64
 11  OR_pitch            6586 non-null   float64
 12  OR_qy               6586 non-null   float64
 13  OR_qz               6586 non-null   float64
 14  OR_roll             6586 non-null   float64
 15  OR_qx       

In [7]:
df_targetexercise_dropped = df_targetexercise.drop(columns = ['is_target', 'target_weight', 'target_repetitions', 'target_name'])

In [ ]:
#df_targetexercise_dropped['timestamp'] = df_targetexercise_dropped.index.round('5s')

In [9]:
df_targetexercise_dropped

,HR_bpm,ACC_z,ACC_y,ACC_x,G_z,G_y,G_x,GYRO_z,GYRO_y,GYRO_x,...,OR_qy,OR_qz,OR_roll,OR_qx,OR_qw,TACC_z,TACC_y,TACC_x,target,timestamp
time,,,,,,,,,,,,,,,,,,,,,
2025-12-12 14:48:31.000,-0.002302,0.834053,0.293834,-1.084356,-5.347019,6.338480,5.210617,-0.103181,-0.800789,0.223799,...,-0.762528,-0.456070,-2.366648,-0.432716,0.100175,-4.816213,6.168937,4.492443,2.0,2025-12-12 14:48:30
2025-12-12 14:48:31.250,0.000000,-0.230473,-0.163688,-0.686758,-6.630318,6.428504,3.210368,-0.250841,-0.899708,0.219461,...,-0.901715,-0.389704,-2.676412,-0.103450,0.137057,-6.710349,6.454169,2.464112,2.0,2025-12-12 14:48:30
2025-12-12 14:48:31.500,0.001108,-0.589627,0.169335,-0.281863,-7.403154,6.251203,1.466870,-0.234084,-0.636644,0.029932,...,-0.936268,-0.344775,-2.982958,0.002996,0.065360,-7.967704,6.322887,0.776519,2.0,2025-12-12 14:48:30
2025-12-12 14:48:31.750,0.001023,0.205271,-0.134327,-0.240932,-7.557937,6.225664,-0.412324,-0.285242,-0.486120,0.132140,...,0.499788,0.171315,1.275434,0.029856,0.021445,-7.472303,5.843408,-0.420542,2.0,2025-12-12 14:48:30
2025-12-12 14:48:32.000,0.000426,-0.400174,-0.029186,-0.335644,-7.710894,5.700234,-1.951509,-0.409745,-0.691150,0.168250,...,0.944382,0.300165,2.931994,0.067321,0.111264,-8.217444,5.723230,-2.001879,2.0,2025-12-12 14:48:30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-27 16:59:21.250,118.986554,3.703231,-1.422605,0.639401,-2.042128,5.731432,7.664852,0.466396,0.127976,-1.165760,...,0.069225,0.404456,-2.151672,0.837231,0.359415,-0.423242,4.973023,6.837707,2.0,2025-12-27 16:59:20
2025-12-27 16:59:21.500,119.202399,3.601724,-2.239705,1.311459,-0.929376,5.257907,8.219067,0.445244,-0.628275,-0.939969,...,-0.034802,0.404487,-1.970044,0.800228,0.439112,0.764016,4.415174,8.081495,2.0,2025-12-27 16:59:20
2025-12-27 16:59:21.750,119.317043,-0.427536,-0.897712,-0.057803,-1.354036,3.948511,8.855087,0.661887,-0.562869,-0.040550,...,-0.202966,0.493118,-1.684121,0.713095,0.453425,-1.516154,3.530094,8.581548,2.0,2025-12-27 16:59:20


In [10]:
#feature generation

extracted_features_tsfresh_targetexercises = extract_features(df_targetexercise_dropped[ [x for x in df_targetexercise_dropped.columns if x!="target"]], column_id='timestamp', column_sort='timestamp')

Feature Extraction: 100%|██████████| 80/80 [00:24<00:00,  3.24it/s]


In [11]:
extracted_features_tsfresh_targetexercises

,OR_yaw__variance_larger_than_standard_deviation,OR_yaw__has_duplicate_max,OR_yaw__has_duplicate_min,OR_yaw__has_duplicate,OR_yaw__sum_values,OR_yaw__abs_energy,OR_yaw__mean_abs_change,OR_yaw__mean_change,OR_yaw__mean_second_derivative_central,OR_yaw__median,...,GYRO_x__fourier_entropy__bins_5,GYRO_x__fourier_entropy__bins_10,GYRO_x__fourier_entropy__bins_100,GYRO_x__permutation_entropy__dimension_3__tau_1,GYRO_x__permutation_entropy__dimension_4__tau_1,GYRO_x__permutation_entropy__dimension_5__tau_1,GYRO_x__permutation_entropy__dimension_6__tau_1,GYRO_x__permutation_entropy__dimension_7__tau_1,GYRO_x__query_similarity_count__query_None__threshold_0.0,GYRO_x__mean_n_absolute_max__number_of_maxima_7
2025-12-12 14:48:30,0.0,0.0,0.0,0.0,2.180027,1.936969,0.227674,-0.191722,0.094750,0.109395,...,1.039721,1.039721,1.386294,1.332179,1.386294,1.098612,0.693147,-0.000000,NaN,NaN
2025-12-12 14:48:35,0.0,0.0,0.0,0.0,1.762141,0.693312,0.128508,0.005574,0.012405,0.163049,...,0.801819,1.418484,2.163956,1.762315,2.512659,2.708050,2.639057,2.564949,NaN,0.367931
2025-12-12 14:48:40,0.0,0.0,0.0,0.0,4.999428,7.689144,0.457641,0.078386,0.002203,0.048273,...,0.639032,1.088900,1.973001,1.676696,2.306669,2.523211,2.639057,2.564949,NaN,0.515953
2025-12-12 14:48:45,0.0,0.0,0.0,0.0,11.416241,11.100929,0.169080,-0.025476,-0.012315,0.628174,...,0.940448,0.940448,1.834372,1.663136,2.395908,2.540036,2.564949,2.484907,NaN,0.362352
2025-12-12 14:48:50,0.0,0.0,0.0,0.0,9.236059,8.960733,0.385441,0.014053,-0.037651,0.170984,...,0.885574,0.885574,2.145842,1.690772,2.293119,2.670120,2.772589,2.708050,NaN,0.378290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-27 16:59:00,1.0,0.0,0.0,0.0,46.906408,139.219665,0.699601,-0.007215,-0.008257,2.561753,...,1.036199,1.468140,2.397895,1.677473,2.505290,2.833213,2.772589,2.708050,NaN,0.245171
2025-12-27 16:59:05,1.0,0.0,0.0,0.0,40.877226,130.539534,1.156281,-0.003477,0.145765,2.517014,...,0.639032,0.940448,2.025326,1.727464,2.599302,2.708050,2.639057,2.564949,NaN,0.249091
2025-12-27 16:59:10,1.0,0.0,0.0,0.0,42.041663,142.027114,0.675430,0.029915,-0.000558,2.525323,...,1.033562,1.594167,2.397895,1.736195,2.582306,2.833213,2.772589,2.708050,NaN,0.429765
2025-12-27 16:59:15,1.0,0.0,0.0,0.0,28.374089,125.187957,1.248710,-0.313040,-0.177732,2.473268,...,0.639032,0.940448,2.163956,1.676696,2.426015,2.708050,2.639057,2.564949,NaN,0.322789


In [ ]:
extracted_features_tsfresh_targetexercises = extracted_features_tsfresh_targetexercises.dropna(axis=1)

In [ ]:
extracted_features_tsfresh_targetexercises

In [ ]:
pd.DataFrame({'column_names': extracted_features_tsfresh_targetexercises.columns}).to_csv('columns_tsfresh_targetexercises.csv', index=False)

In [ ]:
# feature selection
import tsfresh
from tsfresh import select_features


extracted_features_tsfresh_targetexercises.describe()

Link zu tsfresh multiclass example: https://github.com/blue-yonder/tsfresh/blob/main/notebooks/04%20Multiclass%20Selection%20Example.ipynb

In [12]:
#y = df_targetexercise_dropped.target
#ex = np.array([3.0, 4.0, 13.0, 23.0, 26.0, 38.0])
#y = pd.Series(ex)
print(extracted_features_tsfresh_targetexercises.shape)
print(df_targetexercise_dropped.shape)

(379, 15760)
(6586, 22)


In [ ]:
selected_features = tsfresh.select_features(extracted_features_tsfresh_targetexercises,  df_targetexercise_dropped["target"])

In [ ]:
selected_features.

In [ ]:
#takes too long to calculate
df_targetexercise_rolled = roll_time_series (df_targetexercise, column_id='target_name',  column_sort='timestamp')

In [ ]:
df_targetexercise_rolled

In [ ]:
extracted_features_tsfresh_targetexercises_rolled = extract_features(df_targetexercise_rolled, column_id='target_name', column_sort='timestamp')